In [1]:
from pathlib import Path
import json
import sys
sys.path.append(r"C:\Users\F.Turner\Documents\00. Analyses")
import use_funcs
from use_funcs import find_project_root

import pandas as pd

In [2]:
PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "path_config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PATHS = json.load(f)

IMP_DIR = (PROJECT_ROOT / PATHS["imports_dir"]).resolve()
HO_DIR = (PROJECT_ROOT / PATHS["handoff_dir"]).resolve()
EXP_DIR = (PROJECT_ROOT / PATHS["exports_dir"]).resolve()
FIG_DIR = (PROJECT_ROOT / PATHS["figures_dir"]).resolve()
TAB_DIR = (PROJECT_ROOT / PATHS["tables_dir"]).resolve()

for folder in [IMP_DIR, HO_DIR, EXP_DIR, FIG_DIR, TAB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMP_DIR:", IMP_DIR)
print("HO_DIR:", HO_DIR)
print("EXP_DIR:", EXP_DIR)
print("FIG_DIR:", FIG_DIR)
print("TAB_DIR:", TAB_DIR)

PROJECT_ROOT: C:\Users\F.Turner\Documents\00. Analyses\Education Financing
IMP_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Data
HO_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Handoff
EXP_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results
FIG_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results\figures
TAB_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results\tables


In [17]:
# Import all contributing files

cofog_all = pd.read_csv(IMP_DIR / "cofog_all.csv")
cofog_edu = pd.read_csv(IMP_DIR / "cofog_edu.csv")
expenditure = pd.read_csv(IMP_DIR / "expenditure.csv")

lays = pd.read_csv(IMP_DIR / "lays.csv")
pcr = pd.read_csv(IMP_DIR / "pcr.csv")

fsi_summary = pd.read_csv(IMP_DIR / "fsi_summary.csv")
uis_controls = pd.read_csv(IMP_DIR / "uis_controls.csv")
macro = pd.read_csv(IMP_DIR / "macro.csv")
lang_frac = pd.read_csv(IMP_DIR / "lang_frac.csv")

iso3 = pd.read_csv(IMP_DIR / "iso3.csv")

In [18]:
df = pd.merge(lays, pcr, on=["iso3", "year"], how="outer")
df = pd.merge(df, cofog_all, on=["iso3", "year"], how="outer")
df = pd.merge(df, cofog_edu, on=["iso3", "year"], how="outer")
df = pd.merge(df, expenditure, on=["iso3", "year"], how="outer")
df = pd.merge(df, fsi_summary[['iso3', 'year', 'fsi_total']], on=["iso3", "year"], how="outer")
df = pd.merge(df, uis_controls, on=["iso3", "year"], how="outer")
df = pd.merge(df, macro, on=["iso3", "year"], how="outer")
df = pd.merge(df, lang_frac, on=["iso3", "year"], how="outer")

all_data = df.copy()

# **OUTCOME 1: Learning Adjusted Years of Schooling (LAYS)**

In [19]:
lays_data = all_data.dropna(subset=["lays"])

In [21]:
def build_panel_change_dataset(df, start_year, end_year, annual_divisor):
    panel = df[df['year'].isin([start_year, end_year])].copy()
    panel = panel.sort_values(['iso3', 'year']).drop_duplicates(['iso3', 'year'])
    panel = panel[panel.groupby('iso3')['year'].transform('nunique') == 2]

    value_cols = [c for c in panel.columns if c not in ['iso3', 'year', 'income_group']]

    baseline = panel[panel['year'] == start_year].set_index('iso3')[value_cols]
    endline = panel[panel['year'] == end_year].set_index('iso3')[value_cols]

    common_iso3 = baseline.index.intersection(endline.index)
    baseline = baseline.loc[common_iso3]
    endline = endline.loc[common_iso3]

    out = pd.DataFrame(index=common_iso3)

       
    out['start_year'] = start_year
    out['end_year'] = end_year

    for col in value_cols:
        out[f'baseline_{col}'] = baseline[col]
        out[f'delta_{col}'] = (endline[col] - baseline[col]) / annual_divisor

    income_lookup = (
    iso3[["iso3", "wb_income_group"]]
    .drop_duplicates(subset=["iso3"])
    .rename(columns={"wb_income_group": "income_group"})
    )
    out = out.merge(income_lookup, on="iso3", how="left")

    return panel, out.reset_index()

panel1, panel1_change = build_panel_change_dataset(lays_data, 2010, 2017, 7)
panel2, panel2_change = build_panel_change_dataset(lays_data, 2010, 2018, 8)
panel3, panel3_change = build_panel_change_dataset(lays_data, 2010, 2020, 10)
panel4, panel4_change = build_panel_change_dataset(lays_data, 2017, 2020, 3)
panel5, panel5_change = build_panel_change_dataset(lays_data, 2018, 2020, 2)

panel_change_datasets = {
    'panel1_change': panel1_change,
    'panel2_change': panel2_change,
    'panel3_change': panel3_change,
    'panel4_change': panel4_change,
    'panel5_change': panel5_change,
}

panel1_change['panel'] = '1'
panel2_change['panel'] = '2'
panel3_change['panel'] = '3'
panel4_change['panel'] = '4'
panel5_change['panel'] = '5'

all_panels = pd.concat([panel1_change, panel2_change, panel3_change, panel4_change, panel5_change], ignore_index=True)

In [22]:
all_data.to_csv(IMP_DIR / "all_data.csv", index=False)
all_panels.to_csv(IMP_DIR / "lays_panels.csv", index=False)

# **Outcome 2: Primary Completion Rate**

*to be completed*

In [23]:
pcr = all_data.dropna(subset=["pcr_modelled_pct"])
pcr = pcr[pcr.groupby('iso3')['year'].transform('nunique') > 1]

In [24]:
all_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6737 entries, 0 to 6736
Data columns (total 49 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   iso3                                  6737 non-null   str    
 1   year                                  6737 non-null   int64  
 2   lays                                  525 non-null    float64
 3   lays_male                             525 non-null    float64
 4   lays_female                           526 non-null    float64
 5   pcr_modelled_pct                      2625 non-null   float64
 6   pcr_pct                               781 non-null    float64
 7   pcr_wpia                              443 non-null    float64
 8   general_public_services_pct           2267 non-null   float64
 9   defense_pct                           2267 non-null   float64
 10  public_order_safety_pct               2267 non-null   float64
 11  economic_affairs_pct        